# Silver Layer: ERP Location CDC

**Source**: `workspace.bronze.erp_loc_cdc`  
**Target**: `workspace.silver.erp_customer_location_cdc`  
**Primary Key**: `cid` → `customer_number`

**Transformations:**
1. Trim strings
2. Customer ID cleanup (remove hyphens)
3. Country normalization
4. Add country_code column for API joins
5. Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when, regexp_replace
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.erp_loc_a101_cdc"
silver_table = "workspace.silver.erp_customer_location_cdc"

print("✓ Configuration loaded")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Watermark: {watermark}")
else:
    watermark = "1900-01-01"
    print(f"ℹ️  Initial load")

In [0]:
df = spark.table(bronze_table).filter(col("ingestion_timestamp") > watermark)

new_count = df.count()
print(f"📊 New records: {new_count:,}")

if new_count == 0:
    dbutils.notebook.exit("No new data")

In [0]:
# Trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("✓ Trimmed")

In [0]:
# Customer ID cleanup - remove hyphens
df = df.withColumn("cid", regexp_replace(col("cid"), "-", ""))

print("✓ Cleaned customer IDs")

In [0]:
# Country normalization
df = df.withColumn(
    "cntry",
    when(col("cntry") == "DE", "Germany")
    .when(col("cntry").isin("US", "USA"), "United States")
    .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
    .otherwise(col("cntry"))
)

print("✓ Normalized country")

In [0]:
# Add country_code column for API joins
df = df.withColumn(
    "country_code",
    when(col("cntry") == "Germany", "DE")
    .when(col("cntry") == "United States", "US")
    .when(col("cntry") == "United Kingdom", "UK")
    .when(col("cntry") == "France", "FR")
    .when(col("cntry") == "Canada", "CA")
    .when(col("cntry") == "Australia", "AU")
    .when(col("cntry") == "n/a", None)
    .otherwise(None)
)

print("✓ Added country codes")

In [0]:
# Rename columns
rename_map = {
    "cid": "customer_number",
    "cntry": "country"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

print("✓ Renamed columns")
df.limit(5).display()

In [0]:
if not silver_exists:
    print("Creating Silver table...")
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table))
    print(f"✓ Created: {spark.table(silver_table).count():,} records")
else:
    print("Merging into Silver...")
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("target")
      .merge(df.alias("source"), "target.customer_number = source.customer_number")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
    print(f"✓ MERGE complete: {spark.table(silver_table).count():,} total")